# L10 · Dataset Anatomy and Imitation Learning 101

> **Course status:** L10 is `cpu-verified`; the English and Chinese notebooks have both passed this CPU-only data-readback path.

This lab opens the local dataset produced by L09 and follows one evidence path:

```text
identify input → audit metadata → decode synchronized samples → inspect numbers
→ split whole episodes → build an action chunk → compare policy data contracts
```


## Before you run

Reuse the Python environment that completed L09. It should already contain the data-reading dependencies used here: LeRobot, PyArrow, PyAV, NumPy, Matplotlib, and PyTorch.

By default, the notebook reads `ROBO_GENESIS_DATASETS_DIR/l09_banana_demo`. Set `RG101_L10_DATASET_ROOT` before starting the kernel to select another compatible local copy, and optionally set `RG101_L10_REPO_ID` to its logical repository ID. The first code cell checks that exact location before importing the LeRobot reader. It does not scan other directories, download a dataset, or create fallback data.

The dataset itself is opened read-only. This notebook writes only disposable library caches under `ROBO_GENESIS_OUTPUTS_DIR/l10_cache`; it does not initialize Genesis, train a model, create a checkpoint, or measure task success.


In [ ]:
import importlib.metadata
import json
import math
import os
from pathlib import Path

import numpy as np

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.paths import DATASETS_DIR, OUTPUTS_DIR

lesson = load_course_manifest().lesson('L10')
assert lesson.slug == 'dataset-anatomy-and-imitation-learning'
assert lesson.duration_minutes == 90
assert lesson.hardware.value == 'gpu-recommended'
assert lesson.status.value == 'cpu-verified'

configured_root = os.environ.get('RG101_L10_DATASET_ROOT')
dataset_root = (
    Path(configured_root).expanduser().resolve()
    if configured_root
    else (DATASETS_DIR / 'l09_banana_demo').resolve()
)
REPO_ID = os.environ.get('RG101_L10_REPO_ID', f'local/{dataset_root.name}').strip()
if not REPO_ID:
    raise ValueError('RG101_L10_REPO_ID must not be empty')

cache_root = (OUTPUTS_DIR / 'l10_cache').resolve()
for variable, relative in {
    'HF_DATASETS_CACHE': 'huggingface_datasets',
    'XDG_CACHE_HOME': 'xdg',
    'MPLCONFIGDIR': 'matplotlib',
}.items():
    os.environ.setdefault(variable, str(cache_root / relative))
    Path(os.environ[variable]).mkdir(parents=True, exist_ok=True)

required_files = {
    'dataset info': dataset_root / 'meta' / 'info.json',
    'dataset statistics': dataset_root / 'meta' / 'stats.json',
    'task table': dataset_root / 'meta' / 'tasks.parquet',
}
required_groups = {
    'episode metadata': tuple((dataset_root / 'meta' / 'episodes').rglob('*.parquet')),
    'tabular data': tuple((dataset_root / 'data').rglob('*.parquet')),
    'world video': tuple(
        (dataset_root / 'videos' / 'observation.images.world').rglob('*.mp4')
    ),
    'wrist video': tuple(
        (dataset_root / 'videos' / 'observation.images.wrist').rglob('*.mp4')
    ),
}
missing = [label for label, path in required_files.items() if not path.is_file()]
missing += [label for label, paths in required_groups.items() if not paths]
if missing:
    raise FileNotFoundError(
        f'L10 input is incomplete at {dataset_root}. Missing: {", ".join(missing)}. '
        'Run the complete L09 recording notebook, or set RG101_L10_DATASET_ROOT '
        'to an existing compatible dataset. No download or fallback dataset is attempted.'
    )

required_versions = {'lerobot': '0.6.0', 'av': '15.1.0', 'pyarrow': '25.0.0'}
try:
    installed_versions = {
        name: importlib.metadata.version(name) for name in required_versions
    }
except importlib.metadata.PackageNotFoundError as error:
    raise RuntimeError(
        'The L10 data reader dependencies are incomplete. Reuse the environment that '
        'completed L09 and follow the README/COMPATIBILITY.md platform instructions.'
    ) from error
version_mismatches = {
    name: (installed_versions[name], expected)
    for name, expected in required_versions.items()
    if installed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f'Dependency version mismatch: {version_mismatches}')

input_checks = {
    'manifest_contract': lesson.status.value == 'cpu-verified' and lesson.duration_minutes == 90,
    'explicit_input_complete': not missing,
    'reader_versions': not version_mismatches,
}
print(f'Dataset root: {dataset_root}')
print(f'Repository ID: {REPO_ID}')
print('Reader versions:', installed_versions)


## Audit metadata before decoding video

Metadata identifies the dataset, describes every feature, locates logical episode boundaries, and maps those episodes to shared Parquet and MP4 storage files. A storage file can contain more than one episode, so use episode metadata rather than filenames as the source of trajectory identity.

This cell opens only the metadata layer. It reports the codec and backend recorded by the actual input instead of copying an expected value from the lesson.


In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from robo_genesis.record_dataset import JOINT_NAMES

STATE_KEY = 'observation.state'
ACTION_KEY = 'action'
WORLD_KEY = 'observation.images.world'
WRIST_KEY = 'observation.images.wrist'
USER_FEATURE_KEYS = {STATE_KEY, ACTION_KEY, WORLD_KEY, WRIST_KEY}
AUTOMATIC_FIELD_KEYS = {
    'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'
}
EXPECTED_FEATURE_KEYS = USER_FEATURE_KEYS | AUTOMATIC_FIELD_KEYS

metadata = LeRobotDatasetMetadata(REPO_ID, root=dataset_root)
task_names = tuple(str(task) for task in metadata.tasks.index.tolist())
episode_records = [metadata.episodes[index] for index in range(len(metadata.episodes))]
episode_summaries = [
    {
        'episode_index': int(row['episode_index']),
        'tasks': tuple(str(task) for task in row['tasks']),
        'length': int(row['length']),
        'from_index': int(row['dataset_from_index']),
        'to_index': int(row['dataset_to_index']),
    }
    for row in episode_records
]
ordered_bounds = sorted(
    (row['from_index'], row['to_index'], row['length']) for row in episode_summaries
)
episode_bounds_ok = bool(ordered_bounds)
episode_bounds_ok &= ordered_bounds[0][0] == 0
episode_bounds_ok &= ordered_bounds[-1][1] == metadata.total_frames
episode_bounds_ok &= all(end - start == length for start, end, length in ordered_bounds)
episode_bounds_ok &= all(
    left[1] == right[0] for left, right in zip(ordered_bounds, ordered_bounds[1:])
)

data_paths = {
    dataset_root / metadata.get_data_file_path(row['episode_index'])
    for row in episode_summaries
}
video_paths = {
    dataset_root / metadata.get_video_file_path(row['episode_index'], key)
    for row in episode_summaries
    for key in metadata.video_keys
}
video_report = {}
for key in metadata.video_keys:
    specification = metadata.features[key]
    details = specification.get('info') or specification.get('video_info') or {}
    video_report[key] = {
        'stored_shape_hwc': tuple(specification['shape']),
        'codec': details.get('video.codec', 'unreported'),
        'backend': details.get('video.video_backend', 'unreported'),
        'fps': details.get('video.fps', 'unreported'),
    }

metadata_checks = {
    'identity': bool(metadata.info.codebase_version) and metadata.root.resolve() == dataset_root,
    'minimum_size': metadata.total_episodes >= 2
    and metadata.total_frames > 0
    and metadata.total_tasks >= 1
    and metadata.fps > 0,
    'feature_set': set(metadata.features) == EXPECTED_FEATURE_KEYS,
    'joint_names': tuple(metadata.features[STATE_KEY]['names']) == tuple(JOINT_NAMES)
    and tuple(metadata.features[ACTION_KEY]['names']) == tuple(JOINT_NAMES),
    'vector_shapes': tuple(metadata.features[STATE_KEY]['shape']) == (9,)
    and tuple(metadata.features[ACTION_KEY]['shape']) == (9,),
    'camera_contract': set(metadata.video_keys) == {WORLD_KEY, WRIST_KEY}
    and all(tuple(metadata.features[key]['shape'])[-1] == 3 for key in (WORLD_KEY, WRIST_KEY)),
    'tasks_present': len(task_names) == metadata.total_tasks
    and all(name.strip() for name in task_names),
    'episode_boundaries': episode_bounds_ok
    and all(row['length'] > 0 for row in episode_summaries),
    'persistent_files': all(path.is_file() for path in data_paths | video_paths),
    'video_metadata_reported': all(
        report['codec'] != 'unreported' and report['fps'] != 'unreported'
        for report in video_report.values()
    ),
}
failed_metadata = [name for name, passed in metadata_checks.items() if not passed]
if failed_metadata:
    raise AssertionError('Metadata checks failed: ' + ', '.join(failed_metadata))

print(
    f'codebase={metadata.info.codebase_version}; fps={metadata.fps}; '
    f'episodes={metadata.total_episodes}; frames={metadata.total_frames}; '
    f'tasks={metadata.total_tasks}'
)
print('Features:', sorted(metadata.features))
print('Tasks:', task_names)
print('Episode bounds:', json.dumps(episode_summaries, indent=2))
print('Data template:', metadata.data_path)
print('Video template:', metadata.video_path)
print('Actual video metadata:', json.dumps(video_report, indent=2))


## Read and assemble a multimodal training sample

`LeRobotDataset` uses each tabular row as an anchor. When you request `dataset[i]`, the reader returns its state, action, and bookkeeping fields, looks up the task text, and decodes the `world` and `wrist` frames for the same episode and timestamp. The result is one multimodal sample that a later policy preprocessor can consume. Metadata describes stored video as HWC, while the decoded sample exposes CHW float tensors.

The cell below inspects the assembled fields and displays the start, middle, and end of one episode. Each montage column uses one sample, so its world and wrist views share the same episode, frame, and timestamp. Visible progress is evidence that these stored frames are interpretable—not that a policy will use them successfully.


In [ ]:
import matplotlib.pyplot as plt
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(REPO_ID, root=dataset_root, video_backend='pyav')
selected_episode = max(episode_summaries, key=lambda row: row['length'])
if selected_episode['length'] < 4:
    raise ValueError('L10 requires one episode with at least four frames for the montage and H=4 chunk')
selected_episode_id = selected_episode['episode_index']
episode_start = selected_episode['from_index']
episode_length = selected_episode['length']
selected_positions = np.asarray(
    [episode_start, episode_start + episode_length // 2, episode_start + episode_length - 1]
)
selected_samples = [dataset[int(position)] for position in selected_positions]

def tensor_to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def decoded_image(sample, key):
    return tensor_to_numpy(sample[key]).transpose(1, 2, 0)

decoded_images = {
    key: [decoded_image(sample, key) for sample in selected_samples]
    for key in (WORLD_KEY, WRIST_KEY)
}
sample_checks = {
    'row_count': len(dataset) == metadata.total_frames,
    'required_keys': all(
        EXPECTED_FEATURE_KEYS | {'task'} <= set(sample) for sample in selected_samples
    ),
    'numeric_contract': all(
        sample[STATE_KEY].shape == (9,)
        and sample[ACTION_KEY].shape == (9,)
        and sample[STATE_KEY].dtype == torch.float32
        and sample[ACTION_KEY].dtype == torch.float32
        and torch.isfinite(sample[STATE_KEY]).all().item()
        and torch.isfinite(sample[ACTION_KEY]).all().item()
        for sample in selected_samples
    ),
    'task_lookup': all(
        isinstance(sample['task'], str) and sample['task'] in task_names
        for sample in selected_samples
    ),
    'bookkeeping_scalars': all(
        sample['timestamp'].numel() == 1
        and torch.is_floating_point(sample['timestamp'])
        and all(
            sample[key].numel() == 1 and sample[key].dtype == torch.int64
            for key in ('frame_index', 'episode_index', 'index', 'task_index')
        )
        for sample in selected_samples
    ),
    'sample_identity': all(
        int(sample['index']) == int(position)
        and int(sample['frame_index']) == int(position - episode_start)
        and np.isclose(float(sample['timestamp']), (position - episode_start) / metadata.fps)
        for position, sample in zip(selected_positions, selected_samples, strict=True)
    ),
    'same_episode': all(
        int(sample['episode_index']) == selected_episode_id for sample in selected_samples
    ),
    'decoded_image_contract': all(
        sample[key].dtype == torch.float32
        and sample[key].shape
        == (
            metadata.features[key]['shape'][2],
            metadata.features[key]['shape'][0],
            metadata.features[key]['shape'][1],
        )
        and torch.isfinite(sample[key]).all().item()
        and float(sample[key].min()) >= 0.0
        and float(sample[key].max()) <= 1.0
        for sample in selected_samples
        for key in (WORLD_KEY, WRIST_KEY)
    ),
    'nonempty_images': all(
        float(np.std(image)) > 0.0
        for images in decoded_images.values()
        for image in images
    ),
}
failed_samples = [name for name, passed in sample_checks.items() if not passed]
if failed_samples:
    raise AssertionError('Sample checks failed: ' + ', '.join(failed_samples))

first_sample = selected_samples[0]
print(
    f'Assembled sample: episode={int(first_sample["episode_index"])}, '
    f'frame={int(first_sample["frame_index"])}, task={first_sample["task"]!r}'
)
for key in sorted(first_sample):
    value = first_sample[key]
    shape = tuple(value.shape) if hasattr(value, 'shape') else None
    dtype = str(value.dtype) if hasattr(value, 'dtype') else type(value).__name__
    print(f'{key}: type={type(value).__name__}, dtype={dtype}, shape={shape}')

montage, axes = plt.subplots(2, 3, figsize=(12, 6))
for row, key in enumerate((WORLD_KEY, WRIST_KEY)):
    camera_name = key.rsplit('.', 1)[-1]
    for column, (sample, image) in enumerate(
        zip(selected_samples, decoded_images[key], strict=True)
    ):
        axes[row, column].imshow(np.clip(image, 0.0, 1.0))
        axes[row, column].set_title(
            f'{camera_name}: ep={int(sample["episode_index"])}, '
            f'frame={int(sample["frame_index"])}, t={float(sample["timestamp"]):.2f}s'
        )
        axes[row, column].axis('off')
montage.suptitle('Start, middle, and end from one persisted episode')
montage.tight_layout()
plt.show()


## Inspect values, units, and statistics

Read complete state and action columns from the table rather than calling `dataset[i]` for every frame. That avoids decoding two videos merely to draw numeric curves. The first seven channels are arm angles in radians; the final two are finger positions in metres, so they use separate panels.

Statistics can expose non-finite values, implausible ranges, outliers, and nearly constant channels. They do not prove task coverage or generalization. The normalization calculation below is a reversible sanity check for one channel, not a replacement for the saved policy preprocessor used in L12.


In [ ]:
plain_rows = dataset.hf_dataset.with_format(None)
episode_indices = np.asarray(plain_rows['episode_index'], dtype=np.int64)
frame_indices = np.asarray(plain_rows['frame_index'], dtype=np.int64)
global_indices = np.asarray(plain_rows['index'], dtype=np.int64)
state_rows = np.asarray(plain_rows[STATE_KEY], dtype=float)
action_rows = np.asarray(plain_rows[ACTION_KEY], dtype=float)
required_stat_names = {'min', 'max', 'mean', 'std', 'count', 'q01', 'q10', 'q50', 'q90', 'q99'}
vector_stat_names = required_stat_names - {'count'}

def stat_array(feature_key, statistic):
    return np.asarray(metadata.stats[feature_key][statistic], dtype=float)

statistics_checks = {
    'global_index_continuity': np.array_equal(
        global_indices, np.arange(metadata.total_frames)
    ),
    'complete_numeric_arrays': state_rows.shape == action_rows.shape
    == (metadata.total_frames, 9),
    'numeric_values_finite': np.isfinite(state_rows).all() and np.isfinite(action_rows).all(),
    'required_statistics': all(
        required_stat_names <= set(metadata.stats[key]) for key in USER_FEATURE_KEYS
    ),
    'vector_stat_shapes': all(
        stat_array(key, statistic).shape == (9,)
        for key in (STATE_KEY, ACTION_KEY)
        for statistic in vector_stat_names
    ),
    'rgb_stat_shapes': all(
        stat_array(key, statistic).shape == (3, 1, 1)
        for key in (WORLD_KEY, WRIST_KEY)
        for statistic in vector_stat_names
    ),
    'statistics_finite': all(
        np.isfinite(stat_array(key, statistic)).all()
        for key in USER_FEATURE_KEYS
        for statistic in required_stat_names
    ),
    'nonnegative_std': all(
        np.all(stat_array(key, 'std') >= 0.0) for key in USER_FEATURE_KEYS
    ),
    'statistics_count': all(
        int(stat_array(key, 'count').reshape(-1)[0]) == metadata.total_frames
        for key in USER_FEATURE_KEYS
    ),
}

epsilon = 1e-6
arm_channel = 0
arm_mean = float(stat_array(STATE_KEY, 'mean')[arm_channel])
arm_std = float(stat_array(STATE_KEY, 'std')[arm_channel])
arm_scale = max(arm_std, epsilon)
normalized_arm = (state_rows[:, arm_channel] - arm_mean) / arm_scale
restored_arm = normalized_arm * arm_scale + arm_mean
statistics_checks['normalization_round_trip'] = np.isfinite(normalized_arm).all()
statistics_checks['normalization_round_trip'] &= np.allclose(
    restored_arm, state_rows[:, arm_channel]
)
failed_statistics = [name for name, passed in statistics_checks.items() if not passed]
if failed_statistics:
    raise AssertionError('Statistics checks failed: ' + ', '.join(failed_statistics))

for joint_index, joint_name in enumerate(JOINT_NAMES):
    unit = 'rad' if joint_index < 7 else 'm'
    state_range = stat_array(STATE_KEY, 'max')[joint_index] - stat_array(STATE_KEY, 'min')[joint_index]
    action_range = stat_array(ACTION_KEY, 'max')[joint_index] - stat_array(ACTION_KEY, 'min')[joint_index]
    print(
        f'{joint_name}: unit={unit}, state range={state_range:.6g}, '
        f'action range={action_range:.6g}, state std={stat_array(STATE_KEY, "std")[joint_index]:.6g}, '
        f'action std={stat_array(ACTION_KEY, "std")[joint_index]:.6g}'
    )
print(
    f'Channel 0 normalization: mean={arm_mean:.6g}, std={arm_std:.6g}, '
    f'max round-trip error={np.max(np.abs(restored_arm - state_rows[:, arm_channel])):.3g}'
)

episode_mask = episode_indices == selected_episode_id
episode_time = frame_indices[episode_mask] / metadata.fps
figure, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
for joint_index in range(7):
    axes[0].plot(episode_time, state_rows[episode_mask, joint_index], label=f'q{joint_index} state')
    axes[0].plot(episode_time, action_rows[episode_mask, joint_index], '--', alpha=0.75)
axes[0].set_ylabel('arm angle (rad)')
axes[0].set_title(f'Episode {selected_episode_id}: arm state (solid) and action target (dashed)')
axes[0].legend(ncol=4, fontsize=8)
for finger_index in range(2):
    channel = 7 + finger_index
    axes[1].plot(episode_time, state_rows[episode_mask, channel], label=f'finger {finger_index} state')
    axes[1].plot(episode_time, action_rows[episode_mask, channel], '--', label=f'finger {finger_index} target')
axes[1].set_xlabel('episode time (s)')
axes[1].set_ylabel('finger position (m)')
axes[1].set_title('Finger state and action target')
axes[1].legend(ncol=2, fontsize=8)
channels = np.arange(len(JOINT_NAMES))
axes[2].bar(channels - 0.2, stat_array(STATE_KEY, 'std'), width=0.4, label='state std')
axes[2].bar(channels + 0.2, stat_array(ACTION_KEY, 'std'), width=0.4, label='action std')
axes[2].set_xticks(channels, JOINT_NAMES, rotation=35, ha='right')
axes[2].set_ylabel('per-channel std (mixed units)')
axes[2].set_title('Use units above when interpreting per-channel spread')
axes[2].legend()
figure.tight_layout()
plt.show()


## Split complete episodes

Neighboring frames from one rollout share almost the same scene and history. A row-wise random split therefore leaks trajectory information even when no row ID appears twice. This cell mirrors the pinned LeRobot factory rule: group episodes by task and hold out the last `ceil(n × eval_split)` episodes in each group.

With only two same-task episodes, the resulting one/one split demonstrates mechanics; it is not a stable model-selection benchmark. Collection order may also correlate with seeds or conditions, so record the exact episode IDs rather than reporting only a percentage.


In [ ]:
def plan_episode_split(records, eval_split=0.2):
    if not 0.0 < eval_split < 1.0:
        raise ValueError('eval_split must be strictly between 0 and 1')
    task_to_episodes = {}
    for record in records:
        task_key = record['tasks'][0] if record['tasks'] else ''
        task_to_episodes.setdefault(task_key, []).append(record['episode_index'])

    train_ids, eval_ids = [], []
    for episode_ids in task_to_episodes.values():
        n_eval = math.ceil(len(episode_ids) * eval_split)
        train_ids.extend(episode_ids[: len(episode_ids) - n_eval])
        eval_ids.extend(episode_ids[len(episode_ids) - n_eval :])
    if not train_ids:
        raise ValueError(
            f'eval_split={eval_split} leaves no training episodes; collect more episodes per task'
        )
    return train_ids, eval_ids

EVAL_SPLIT = 0.2
train_episode_ids, eval_episode_ids = plan_episode_split(episode_summaries, EVAL_SPLIT)
all_episode_ids = [row['episode_index'] for row in episode_summaries]
summary_by_id = {row['episode_index']: row for row in episode_summaries}
split_frame_totals = {
    'train': sum(summary_by_id[index]['length'] for index in train_episode_ids),
    'eval': sum(summary_by_id[index]['length'] for index in eval_episode_ids),
}
split_checks = {
    'both_sides_nonempty': bool(train_episode_ids) and bool(eval_episode_ids),
    'episode_ids_disjoint': set(train_episode_ids).isdisjoint(eval_episode_ids),
    'full_episode_coverage': set(train_episode_ids) | set(eval_episode_ids)
    == set(all_episode_ids),
    'frame_coverage': sum(split_frame_totals.values()) == metadata.total_frames,
}
failed_split = [name for name, passed in split_checks.items() if not passed]
if failed_split:
    raise AssertionError('Episode split checks failed: ' + ', '.join(failed_split))

split_rows = []
for record in episode_summaries:
    split = 'train' if record['episode_index'] in train_episode_ids else 'eval'
    split_rows.append({**record, 'split': split})
    print(
        f'episode={record["episode_index"]} split={split} tasks={record["tasks"]} '
        f'length={record["length"]} bounds=[{record["from_index"]}, {record["to_index"]})'
    )
print('Frame totals:', split_frame_totals)
if len(all_episode_ids) == 2:
    print('MECHANICS ONLY: two episodes do not form a stable benchmark.')

split_figure, split_axis = plt.subplots(figsize=(10, max(2.5, 0.7 * len(split_rows))))
colors = {'train': '#3b82f6', 'eval': '#22c55e'}
for y, record in enumerate(split_rows):
    split_axis.barh(y, record['length'], color=colors[record['split']], alpha=0.75)
    split_axis.text(
        record['length'] / 2, y, f'{record["split"]}: {record["length"]} frames',
        ha='center', va='center'
    )
split_axis.set_yticks(range(len(split_rows)), [f'episode {row["episode_index"]}' for row in split_rows])
split_axis.set_xlabel('episode length (frames)')
split_axis.set_title('Whole-episode train/eval assignment')
split_figure.tight_layout()
plt.show()


## Turn one behavior-cloning target into an action chunk

Behavior cloning pairs the current observation with the expert action recorded at that decision time. An action-chunk policy requests several future expert actions at dataset-FPS offsets. The reader must keep that window inside the current episode.

For `H=4`, inspect one interior frame and the episode's final frame. The final row retains a fixed `(4, 9)` shape by copying the boundary action, while `action_is_pad` marks the three copied rows so a later training loss can exclude them. `H/fps` is the nominal duration represented by four samples; `(H-1)/fps` is the offset of the final sampled target.


In [ ]:
H = 4
fps = metadata.fps
delta_timestamps = {
    'action': [i / fps for i in range(H)],
}
chunk_reader = LeRobotDataset(
    REPO_ID,
    root=dataset_root,
    episodes=[selected_episode_id],
    delta_timestamps=delta_timestamps,
    video_backend='pyav',
)
middle_position = min(len(chunk_reader) // 2, len(chunk_reader) - H)
last_position = len(chunk_reader) - 1
middle_chunk = chunk_reader[middle_position]
final_chunk = chunk_reader[last_position]
middle_mask = tensor_to_numpy(middle_chunk['action_is_pad']).astype(bool)
final_mask = tensor_to_numpy(final_chunk['action_is_pad']).astype(bool)
middle_actions = tensor_to_numpy(middle_chunk[ACTION_KEY])
final_actions = tensor_to_numpy(final_chunk[ACTION_KEY])
nominal_horizon_s = H / fps
final_target_offset_s = (H - 1) / fps

chunk_checks = {
    'offset_alignment': np.allclose(
        delta_timestamps[ACTION_KEY], np.arange(H, dtype=float) / fps
    ),
    'fixed_shapes': middle_actions.shape == final_actions.shape == (H, 9),
    'middle_unpadded': middle_mask.shape == (H,) and not middle_mask.any(),
    'final_boundary_mask': np.array_equal(
        final_mask, np.array([False, True, True, True])
    ),
    'boundary_values_copied': np.allclose(
        final_actions, np.repeat(final_actions[:1], H, axis=0)
    ),
    'same_episode': int(middle_chunk['episode_index']) == selected_episode_id
    and int(final_chunk['episode_index']) == selected_episode_id,
    'finite_valid_targets': np.isfinite(middle_actions[~middle_mask]).all()
    and np.isfinite(final_actions[~final_mask]).all(),
}
failed_chunks = [name for name, passed in chunk_checks.items() if not passed]
if failed_chunks:
    raise AssertionError('Action-chunk checks failed: ' + ', '.join(failed_chunks))

print('Action offsets (s):', delta_timestamps[ACTION_KEY])
print(f'Nominal H/fps horizon: {nominal_horizon_s:.3f} s')
print(f'Final sampled-target offset: {final_target_offset_s:.3f} s')
print('Middle mask:', middle_mask.tolist())
print('Final mask:', final_mask.tolist())

mask_figure, mask_axis = plt.subplots(figsize=(8, 2.5))
mask_image = np.vstack([middle_mask, final_mask]).astype(int)
mask_axis.imshow(mask_image, cmap='Reds', vmin=0, vmax=1, aspect='auto')
mask_axis.set_xticks(range(H), [f'a_(t+{offset})' for offset in range(H)])
mask_axis.set_yticks([0, 1], ['middle frame', 'final frame'])
mask_axis.set_title('action_is_pad: white = valid target, red = padding')
for row in range(mask_image.shape[0]):
    for column in range(mask_image.shape[1]):
        mask_axis.text(column, row, bool(mask_image[row, column]), ha='center', va='center')
mask_figure.tight_layout()
plt.show()


## Compare the ACT and SmolVLA data contracts

Both policies consume the same raw course dataset and same-episode action targets. The current ACT-from-configuration path adapts directly to the dataset camera keys. The SmolVLA preset also consumes task text and renames `world`/`wrist` to the camera names expected by its pretrained base.

The rename is a saved preprocessing adapter, not a request to rename files. This cell reads the repository preset without importing or allocating either model. Architecture, optimization, and checkpoint behavior remain in L12.


In [ ]:
from robo_genesis.train_policy import PRESETS

expected_smolvla_rename = {
    WORLD_KEY: 'observation.images.camera1',
    WRIST_KEY: 'observation.images.camera2',
}
act_rename = dict(PRESETS['act'].get('rename_map', {}))
smolvla_rename = dict(PRESETS['smolvla'].get('rename_map', {}))
policy_contracts = [
    {
        'policy': 'ACT',
        'raw_observations': (STATE_KEY, WORLD_KEY, WRIST_KEY),
        'task_text': 'not required by this course path',
        'target': f'same-episode action chunk ({H}, 9)',
        'rename_map': act_rename,
    },
    {
        'policy': 'SmolVLA',
        'raw_observations': (STATE_KEY, WORLD_KEY, WRIST_KEY),
        'task_text': 'required',
        'target': f'same-episode action chunk ({H}, 9)',
        'rename_map': smolvla_rename,
    },
]
policy_checks = {
    'raw_fields_available': {STATE_KEY, WORLD_KEY, WRIST_KEY} <= set(metadata.features),
    'task_text_available': bool(task_names) and all(name.strip() for name in task_names),
    'act_uses_raw_camera_keys': act_rename == {},
    'smolvla_camera_adapter': smolvla_rename == expected_smolvla_rename,
    'same_chunk_target': all(
        contract['target'] == f'same-episode action chunk ({H}, 9)'
        for contract in policy_contracts
    ),
}
failed_policy = [name for name, passed in policy_checks.items() if not passed]
if failed_policy:
    raise AssertionError('Policy data-contract checks failed: ' + ', '.join(failed_policy))

for contract in policy_contracts:
    print(json.dumps(contract, indent=2))
print('No model was imported or allocated.')


## Summarize only the evidence produced here

A passing report means that this local dataset satisfied the stated identity, structure, decoding, numeric, split, action-window, and adapter checks. It does not mean that the demonstrations are sufficient, that a policy can fit them, or that the robot succeeds in closed loop.

Offline evaluation compares predictions with held-out expert actions. Closed-loop evaluation lets each policy action change the next observation and measures a task predicate in Genesis. L13, not this notebook, provides that final evidence.


In [ ]:
section_checks = {
    'input': input_checks,
    'metadata': metadata_checks,
    'sample_and_video': sample_checks,
    'statistics': statistics_checks,
    'episode_split': split_checks,
    'action_chunk': chunk_checks,
    'policy_contracts': policy_checks,
}
failed = [
    f'{section}.{name}'
    for section, checks in section_checks.items()
    for name, passed in checks.items()
    if not passed
]
if failed:
    raise AssertionError('L10 checks failed: ' + ', '.join(failed))

print(
    f'Validated {metadata.total_episodes} episodes / {metadata.total_frames} frames at '
    f'{metadata.fps} FPS from {dataset_root}'
)
print(f'Train episodes: {train_episode_ids}; eval episodes: {eval_episode_ids}')
print(f'Action chunk: H={H}, shape={(H, 9)}, final mask={final_mask.tolist()}')
print('Training: NOT RUN')
print('Checkpoint: NOT CREATED')
print('Genesis rollout: NOT RUN')
print('Task success rate: NOT MEASURED')
print('L10 CHECK: PASSED')


## Checkpoint and one-variable exercise

Using the report and plots above, explain:

1. why one Parquet or MP4 chunk can contain several logical episodes;
2. why metadata HWC and decoded CHW are both correct;
3. why whole-episode numeric plots should not decode every video frame;
4. why arm and finger values need separate units;
5. how a random frame split leaks trajectory information;
6. why copied boundary actions are valid only when the padding mask is honored;
7. why SmolVLA needs the current camera-key adapter while ACT does not; and
8. why finite data and low held-out loss cannot establish closed-loop success.

For the one-variable exercise, keep the dataset, selected episode, FPS, feature schema, and split fixed. Change only `H` to 2, 4, and 8. Before constructing each reader, predict the action shape, `H/fps`, `(H-1)/fps`, and final-frame mask; then compare the prediction with the reader output and count only unmasked targets. Do not train three policies or infer which horizon would achieve the best task success.
